In [5]:
# BOARD COMPOSITION & ESG PERFORMANCE — ECONOMETRIC ANALYSIS
# ============================================================
#
# Research Question:
#   How do specific board characteristics influence a firm's ESG
#   score, and how do these relationships differ across sectors?
#
# Hypotheses:
#   H1 — Higher female board representation → higher ESG scores (both sectors)
#   H2 — Board independence → higher ESG scores (both sectors)
#   H3 — Board size: positive in Tech; negative/insignificant in Energy
#   H4 — CEO Duality: nuanced in Tech; role separation helps in Energy
#
# OLS Model:
#   ESG_Score = β0 + β1(Board_Size) + β2(Pct_Independent_Directors)
#             + β3(Pct_Women_on_Board) + β4(CEO_Duality)
#             + β5(Industry_Dummy) + ε
#
# Data: Bloomberg Terminal — 10 firms × 5 years (2019–2023) = 50 obs
#       5 Technology: MSFT, NVDA, AAPL, GOOGL, SAP
#       5 Energy:     BP, SHEL, TTE, XOM, CVX
# ============================================================


In [6]:
# ── STEP 0: Import libraries ─────────────────────────────────
# These are the only four packages needed for this analysis.
# Install them first if needed: pip install pandas statsmodels scipy

import pandas as pd               # for loading and handling data
import numpy as np                # for numerical operations
import statsmodels.api as sm      # for OLS regression
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy import stats           # for Pearson correlation
import warnings
warnings.filterwarnings('ignore')

In [7]:
# ── STEP 1: Load the cleaned panel dataset ────────────────────
# The file 'ESG_Board_Panel_Dataset.xlsx' must be in the same
# folder as this script. 'header=1' tells pandas to use the
# second row (index 1) as column names, skipping the group label row.

df = pd.read_excel(
    'ESG_Board_Panel_Dataset.xlsx',
    sheet_name='Panel Data',
    header=1
)

# Rename all columns to short, clean names for the analysis
df.columns = [
    'Company', 'Ticker', 'Industry', 'Year', 'Industry_Dummy',
    'ESG_Score', 'Environmental_Score', 'Social_Score',
    'Governance_Score', 'ESG_Disclosure_Score',
    'Board_Size', 'Pct_Independent_Directors',
    'Pct_Women_on_Board', 'Board_Average_Age',
    'CEO_Duality', 'Num_Board_Meetings',
    'Board_Meeting_Attendance_Pct', 'Pct_Female_Executives',
    'Audit_Committee_Size'
]

# Replace any 'N/A' text with actual NaN (missing), then force
# all non-text columns to be read as numbers
df.replace('N/A', np.nan, inplace=True)
num_cols = df.columns.drop(['Company', 'Ticker', 'Industry'])
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')

print(f"\nDataset loaded successfully.")
print(f"Observations: {df.shape[0]}  |  Variables: {df.shape[1]}")
print(f"Years: {sorted(df['Year'].unique())}")
print(f"Sector split: {df['Industry'].value_counts().to_dict()}")

# Define which variables play which role
DV       = 'ESG_Score'                            # dependent variable
IVs      = ['Board_Size',
            'Pct_Independent_Directors',
            'Pct_Women_on_Board',
            'CEO_Duality']                         # independent variables
CONTROL  = 'Industry_Dummy'                        # control variable
ALL_VARS = IVs + [CONTROL, DV]

# Split into sector subsets — needed for H3 and H4 sector tests
tech   = df[df['Industry'] == 'Tech'].copy()       # 25 observations
energy = df[df['Industry'] == 'Energy'].copy()     # 25 observations


Dataset loaded successfully.
Observations: 50  |  Variables: 19
Years: [2019, 2020, 2021, 2022, 2023]
Sector split: {'Tech': 25, 'Energy': 25}


In [8]:
# ============================================================
# SECTION 1 — DESCRIPTIVE STATISTICS
# ============================================================
# Descriptive stats give a picture of the data before any
# modelling — showing central tendency and spread by sector.

print("=" * 65)
print("  BOARD COMPOSITION & ESG — ECONOMETRIC ANALYSIS")
print("=" * 65)

print("\n" + "=" * 65)
print("  SECTION 1: DESCRIPTIVE STATISTICS")
print("=" * 65)

def desc_stats(data, label):
    """Print mean, std dev, min, and max for all model variables."""
    subset = data[ALL_VARS].describe().T
    subset.columns = ['N', 'Mean', 'Std Dev', 'Min', '25%', '50%', '75%', 'Max']
    subset = subset[['N', 'Mean', 'Std Dev', 'Min', 'Max']].round(3)
    print(f"\n  [{label}]  (n = {len(data)})")
    print(subset.to_string())

desc_stats(df,     "FULL SAMPLE (Pooled)")
desc_stats(tech,   "TECHNOLOGY SECTOR")
desc_stats(energy, "ENERGY SECTOR")

# Sector mean comparison table
print("\n  [SECTOR MEAN COMPARISON]")
print(f"  {'Variable':<32} {'Tech Mean':>10} {'Energy Mean':>12} {'Difference':>12}")
print("  " + "-" * 68)
for var in ALL_VARS:
    t_mean = tech[var].mean()
    e_mean = energy[var].mean()
    diff   = e_mean - t_mean
    print(f"  {var:<32} {t_mean:>10.3f} {e_mean:>12.3f} {diff:>+12.3f}")

  BOARD COMPOSITION & ESG — ECONOMETRIC ANALYSIS

  SECTION 1: DESCRIPTIVE STATISTICS

  [FULL SAMPLE (Pooled)]  (n = 50)
                              N    Mean  Std Dev    Min    Max
Board_Size                 50.0  12.100    2.525   7.00  18.00
Pct_Independent_Directors  50.0  79.848   13.546  50.00  92.31
Pct_Women_on_Board         50.0  37.294    9.865  16.67  58.33
CEO_Duality                50.0   0.360    0.485   0.00   1.00
Industry_Dummy             50.0   0.500    0.505   0.00   1.00
ESG_Score                  50.0   5.562    0.833   3.46   7.05

  [TECHNOLOGY SECTOR]  (n = 25)
                              N   Mean  Std Dev    Min    Max
Board_Size                 25.0  12.28    3.410   7.00  18.00
Pct_Independent_Directors  25.0  78.35   16.193  50.00  92.31
Pct_Women_on_Board         25.0  32.85    9.903  16.67  50.00
CEO_Duality                25.0   0.12    0.332   0.00   1.00
Industry_Dummy             25.0   0.00    0.000   0.00   0.00
ESG_Score                  25.0 

In [9]:
# ============================================================
# SECTION 2 — CORRELATION MATRIX
# ============================================================
# A Pearson correlation matrix checks for:
#   (a) Initial linear associations between IVs and the DV
#   (b) Multicollinearity between IVs (|r| > 0.70 = concern)

print("\n" + "=" * 65)
print("  SECTION 2: PEARSON CORRELATION MATRIX")
print("=" * 65)
print("  Values range from -1 to +1.")
print("  |r| > 0.70 between two IVs = multicollinearity risk.\n")

corr_matrix = df[ALL_VARS].corr(method='pearson').round(3)
print(corr_matrix.to_string())

# Flag high IV-to-IV correlations
print("\n  Multicollinearity screening (|r| > 0.70 between IVs):")
flag_found = False
for i, v1 in enumerate(IVs):
    for v2 in IVs[i + 1:]:
        r = abs(corr_matrix.loc[v1, v2])
        if r > 0.70:
            print(f"  ⚠  {v1} ↔ {v2}: r = {corr_matrix.loc[v1, v2]:.3f}")
            flag_found = True
if not flag_found:
    print("  ✓  No IV pair exceeds r = 0.70. No multicollinearity concern.")

# IV–DV correlation significance tests
print("\n  Bivariate correlations with ESG_Score (Pearson r):")
for iv in IVs + [CONTROL]:
    clean = df[[iv, DV]].dropna()
    r, p  = stats.pearsonr(clean[iv], clean[DV])
    sig   = "***" if p < 0.01 else ("**" if p < 0.05 else ("*" if p < 0.10 else "n.s."))
    print(f"  {iv:<34} r = {r:+.3f}   p = {p:.4f}   {sig}")

print("  Codes: *** p<0.01  ** p<0.05  * p<0.10  n.s. = not significant")


  SECTION 2: PEARSON CORRELATION MATRIX
  Values range from -1 to +1.
  |r| > 0.70 between two IVs = multicollinearity risk.

                           Board_Size  Pct_Independent_Directors  Pct_Women_on_Board  CEO_Duality  Industry_Dummy  ESG_Score
Board_Size                      1.000                     -0.660               0.358       -0.030          -0.072     -0.023
Pct_Independent_Directors      -0.660                      1.000              -0.350        0.243           0.112      0.202
Pct_Women_on_Board              0.358                     -0.350               1.000        0.238           0.455      0.207
CEO_Duality                    -0.030                      0.243               0.238        1.000           0.500      0.018
Industry_Dummy                 -0.072                      0.112               0.455        0.500           1.000      0.390
ESG_Score                      -0.023                      0.202               0.207        0.018           0.390      1.00

In [10]:
# ============================================================
# SECTION 3 — VARIANCE INFLATION FACTOR (VIF)
# ============================================================
# VIF is a more formal multicollinearity test than correlation.
# Each IV is regressed on all other IVs; a high VIF means that
# IV is largely predictable from the others.
# Rule of thumb: VIF < 5 = acceptable.

print("\n" + "=" * 65)
print("  SECTION 3: MULTICOLLINEARITY — VARIANCE INFLATION FACTORS")
print("=" * 65)
print("  VIF < 5: acceptable  |  VIF 5–10: moderate  |  VIF > 10: severe\n")

# Build the X matrix (IVs + control, no DV)
vif_data = df[IVs + [CONTROL]].dropna()
X_vif    = sm.add_constant(vif_data)

print(f"  {'Variable':<34} {'VIF':>8}")
print("  " + "-" * 46)
for i, col in enumerate(X_vif.columns[1:], 1):    # skip the constant column
    vif_val = variance_inflation_factor(X_vif.values, i)
    status  = "  ✓ OK" if vif_val < 5 else "  ⚠ concern"
    print(f"  {col:<34} {vif_val:>8.3f}{status}")


  SECTION 3: MULTICOLLINEARITY — VARIANCE INFLATION FACTORS
  VIF < 5: acceptable  |  VIF 5–10: moderate  |  VIF > 10: severe

  Variable                                VIF
  ----------------------------------------------
  Board_Size                            1.920  ✓ OK
  Pct_Independent_Directors             2.095  ✓ OK
  Pct_Women_on_Board                    1.674  ✓ OK
  CEO_Duality                           1.472  ✓ OK
  Industry_Dummy                        1.701  ✓ OK


In [11]:
# ============================================================
# SECTION 4 — OLS REGRESSION: POOLED FULL SAMPLE
# ============================================================
# OLS finds the linear combination of IVs that best predicts
# the DV by minimising the sum of squared residuals.
# sm.add_constant() adds the intercept (β0) to the model.

print("\n" + "=" * 65)
print("  SECTION 4: OLS REGRESSION — POOLED FULL SAMPLE (n = 50)")
print("=" * 65)
print("  Model: ESG_Score ~ Board_Size + Pct_Independent_Directors")
print("                   + Pct_Women_on_Board + CEO_Duality")
print("                   + Industry_Dummy\n")

reg_data = df[ALL_VARS].dropna()                   # drop any rows with missing values
X_pool   = sm.add_constant(reg_data[IVs + [CONTROL]])
y_pool   = reg_data[DV]

ols_pool = sm.OLS(y_pool, X_pool).fit()
print(ols_pool.summary())

# ── Breusch-Pagan test for heteroskedasticity ────────────────
# This checks whether the variance of the residuals is constant
# (homoskedasticity). A significant p-value (< 0.05) means the
# residual variance changes across observations — violating one
# of the OLS assumptions. If detected, use robust standard errors.

bp_lm, bp_p, _, _ = het_breuschpagan(ols_pool.resid, X_pool)
print("\n  Breusch-Pagan Test for Heteroskedasticity:")
print(f"  LM statistic = {bp_lm:.4f}   p-value = {bp_p:.4f}")
if bp_p < 0.05:
    print("  ⚠  Heteroskedasticity detected (p < 0.05).")
    print("     Reporting HC3 robust standard errors below as robustness check.")
else:
    print("  ✓  No significant heteroskedasticity. OLS standard errors are valid.")

# ── Robustness check: OLS with HC3 robust standard errors ─────
# cov_type='HC3' corrects standard errors for heteroskedasticity
# without changing the coefficients themselves.

print("\n  [ROBUSTNESS CHECK] OLS with HC3 Robust Standard Errors:\n")
ols_robust = sm.OLS(y_pool, X_pool).fit(cov_type='HC3')
print(ols_robust.summary())



  SECTION 4: OLS REGRESSION — POOLED FULL SAMPLE (n = 50)
  Model: ESG_Score ~ Board_Size + Pct_Independent_Directors
                   + Pct_Women_on_Board + CEO_Duality
                   + Industry_Dummy

                            OLS Regression Results                            
Dep. Variable:              ESG_Score   R-squared:                       0.292
Model:                            OLS   Adj. R-squared:                  0.211
Method:                 Least Squares   F-statistic:                     3.621
Date:                Wed, 12 Aug 2026   Prob (F-statistic):            0.00789
Time:                        21:44:56   Log-Likelihood:                -52.698
No. Observations:                  50   AIC:                             117.4
Df Residuals:                      44   BIC:                             128.9
Df Model:                           5                                         
Covariance Type:            nonrobust                                         


In [12]:
# ============================================================
# SECTION 5 — SECTOR-LEVEL OLS (for H3 and H4)
# ============================================================
# Running separate OLS models within each sector lets us test
# whether Board_Size and CEO_Duality behave differently by sector.
# Industry_Dummy is excluded here — there is only one sector per model.

print("\n" + "=" * 65)
print("  SECTION 5: SECTOR-LEVEL OLS — TECHNOLOGY vs ENERGY")
print("=" * 65)
print("  (Industry_Dummy excluded — single-sector regressions)\n")

IVs_sector = ['Board_Size', 'Pct_Independent_Directors',
               'Pct_Women_on_Board', 'CEO_Duality']

def run_sector_ols(data, label):
    """Run OLS for a single sector and print a compact results table."""
    clean = data[IVs_sector + [DV]].dropna()
    X_sec = sm.add_constant(clean[IVs_sector])
    y_sec = clean[DV]
    model = sm.OLS(y_sec, X_sec).fit()

    # Build a tidy summary DataFrame
    sig_codes = ['***' if p < 0.01 else '**' if p < 0.05 else '*'
                 if p < 0.10 else 'n.s.' for p in model.pvalues]
    results = pd.DataFrame({
        'Coefficient':  model.params.round(4),
        'Std Error':    model.bse.round(4),
        't-stat':       model.tvalues.round(3),
        'p-value':      model.pvalues.round(4),
        'Significance': sig_codes
    })

    print(f"\n  ── {label} (n = {len(clean)}) ──")
    print(results.to_string())
    print(f"\n  R-squared: {model.rsquared:.4f}   "
          f"Adj. R²: {model.rsquared_adj:.4f}   "
          f"F-stat: {model.fvalue:.3f} (p = {model.f_pvalue:.4f})")
    return model

model_tech   = run_sector_ols(tech,   "TECHNOLOGY SECTOR OLS")
model_energy = run_sector_ols(energy, "ENERGY SECTOR OLS")


  SECTION 5: SECTOR-LEVEL OLS — TECHNOLOGY vs ENERGY
  (Industry_Dummy excluded — single-sector regressions)


  ── TECHNOLOGY SECTOR OLS (n = 25) ──
                           Coefficient  Std Error  t-stat  p-value Significance
const                          -0.7759     2.3518  -0.330   0.7449         n.s.
Board_Size                      0.1695     0.0717   2.365   0.0283           **
Pct_Independent_Directors       0.0533     0.0179   2.981   0.0074          ***
Pct_Women_on_Board             -0.0062     0.0252  -0.245   0.8089         n.s.
CEO_Duality                    -0.3224     0.6801  -0.474   0.6406         n.s.

  R-squared: 0.3957   Adj. R²: 0.2748   F-stat: 3.273 (p = 0.0322)

  ── ENERGY SECTOR OLS (n = 25) ──
                           Coefficient  Std Error  t-stat  p-value Significance
const                           5.1576     1.1041   4.671   0.0001          ***
Board_Size                      0.1165     0.0601   1.937   0.0669            *
Pct_Independent_Directors

In [13]:
# ============================================================
# SECTION 6 — HYPOTHESIS EVALUATION SUMMARY
# ============================================================
# Collect coefficients and p-values to assess each hypothesis.

print("\n" + "=" * 65)
print("  SECTION 6: HYPOTHESIS EVALUATION SUMMARY")
print("=" * 65)

def sig_label(p):
    return "***" if p < 0.01 else ("**" if p < 0.05 else ("*" if p < 0.10 else "n.s."))

# Pooled model results
params = ols_pool.params
pvals  = ols_pool.pvalues

c_women = params['Pct_Women_on_Board'];    p_women = pvals['Pct_Women_on_Board']
c_indep = params['Pct_Independent_Directors']; p_indep = pvals['Pct_Independent_Directors']

# Sector-level results
c_bsize_t = model_tech.params['Board_Size'];   p_bsize_t = model_tech.pvalues['Board_Size']
c_bsize_e = model_energy.params['Board_Size']; p_bsize_e = model_energy.pvalues['Board_Size']
c_dual_t  = model_tech.params['CEO_Duality'];  p_dual_t  = model_tech.pvalues['CEO_Duality']
c_dual_e  = model_energy.params['CEO_Duality'];p_dual_e  = model_energy.pvalues['CEO_Duality']

print(f"""
  H1 — Female Board Representation → Positive ESG Effect
       Pooled β = {c_women:+.4f}   p = {p_women:.4f}   {sig_label(p_women)}
       {'✓  SUPPORTED' if c_women > 0 and p_women < 0.10 else '✗  NOT SUPPORTED (direction correct but insignificant pooled)'}
       Note: Significant in Energy sub-model (β = +0.023, p = 0.013**)

  H2 — Board Independence → Positive ESG Effect
       Pooled β = {c_indep:+.4f}   p = {p_indep:.4f}   {sig_label(p_indep)}
       {'✓  SUPPORTED' if c_indep > 0 and p_indep < 0.10 else '✗  NOT SUPPORTED'}

  H3 — Board Size: Positive (Tech) / Negative or Insignificant (Energy)
       Tech   β = {c_bsize_t:+.4f}   p = {p_bsize_t:.4f}   {sig_label(p_bsize_t)}
       Energy β = {c_bsize_e:+.4f}   p = {p_bsize_e:.4f}   {sig_label(p_bsize_e)}
       ~  PARTIALLY SUPPORTED — both sectors show positive direction;
          Energy effect is only marginal (p = 0.067), not negative.

  H4 — CEO Duality: Nuanced (Tech) / Role Separation Positive (Energy)
       Tech   β = {c_dual_t:+.4f}   p = {p_dual_t:.4f}   {sig_label(p_dual_t)}
       Energy β = {c_dual_e:+.4f}   p = {p_dual_e:.4f}   {sig_label(p_dual_e)}
       ✓  SUPPORTED — duality significantly reduces ESG in Energy (p < 0.001);
          effect is statistically insignificant (and smaller) in Tech,
          consistent with the 'nuanced' prediction.

  Significance codes: *** p<0.01  ** p<0.05  * p<0.10  n.s. = not significant
""")


# ── Key diagnostic note ───────────────────────────────────────
print("  DIAGNOSTIC NOTE:")
print(f"  Durbin-Watson statistic (pooled OLS) = {sm.stats.stattools.durbin_watson(ols_pool.resid):.3f}")
print("  A value close to 2.0 is ideal; values near 0 indicate positive")
print("  autocorrelation. With panel data and n=50, this is an expected")
print("  limitation of pooled OLS — noted as a study limitation alongside")
print("  the absence of financial control variables and firm fixed effects.")

print("\n" + "=" * 65)
print("  END OF ANALYSIS")
print("=" * 65)


  SECTION 6: HYPOTHESIS EVALUATION SUMMARY

  H1 — Female Board Representation → Positive ESG Effect
       Pooled β = +0.0125   p = 0.3713   n.s.
       ✗  NOT SUPPORTED (direction correct but insignificant pooled)
       Note: Significant in Energy sub-model (β = +0.023, p = 0.013**)

  H2 — Board Independence → Positive ESG Effect
       Pooled β = +0.0276   p = 0.0188   **
       ✓  SUPPORTED

  H3 — Board Size: Positive (Tech) / Negative or Insignificant (Energy)
       Tech   β = +0.1695   p = 0.0283   **
       Energy β = +0.1165   p = 0.0669   *
       ~  PARTIALLY SUPPORTED — both sectors show positive direction;
          Energy effect is only marginal (p = 0.067), not negative.

  H4 — CEO Duality: Nuanced (Tech) / Role Separation Positive (Energy)
       Tech   β = -0.3224   p = 0.6406   n.s.
       Energy β = -0.5912   p = 0.0001   ***
       ✓  SUPPORTED — duality significantly reduces ESG in Energy (p < 0.001);
          effect is statistically insignificant (and smalle